# Xây Dựng Ứng Dụng Tạo Ảnh (Azure OpenAI)

Có nhiều hơn thế về LLMs ngoài việc tạo văn bản. Bạn cũng có thể tạo ảnh từ mô tả văn bản. Ảnh như một hình thức truyền tải rất hữu ích trong MedTech, kiến trúc, du lịch, phát triển game, marketing và nhiều lĩnh vực khác. Trong bài học này, chúng ta sẽ làm việc với các mô hình **GPT Image** ngày nay trên Microsoft Foundry.

## Mục tiêu học tập

Cuối bài học này, bạn sẽ có thể:

- Giải thích tạo ảnh là gì và nó hữu ích ở đâu.
- Hiểu về dòng mô hình `gpt-image` và cách nó khác biệt so với các mô hình DALL·E cũ.
- Xây dựng ứng dụng tạo ảnh và chỉnh sửa ảnh.

## Tạo ảnh là gì?

Mô hình tạo ảnh tạo ra hình ảnh từ một đoạn văn bản mô tả. Các mô hình hiện đại như `gpt-image` học mối quan hệ giữa văn bản và ảnh trong quá trình huấn luyện, sau đó từng bước biến nhiễu ngẫu nhiên thành hình ảnh phù hợp với mô tả của bạn.

Hai dòng mô hình ảnh nổi tiếng là:

- **`gpt-image` (OpenAI)** — thế hệ hiện tại được sử dụng trong bài học này. Nó hỗ trợ tạo ảnh từ văn bản và chỉnh sửa ảnh (vẽ chồng với mặt nạ).
- **Midjourney** — một mô hình bên thứ ba phổ biến với dịch vụ riêng và quy trình làm việc dựa trên Discord.

> Các mô hình ảnh OpenAI cũ hơn — **DALL·E 2** và **DALL·E 3** — là các mô hình kế thừa. DALL·E 3 không còn cho triển khai mới, và các tính năng như `create_variation` chỉ tồn tại trong DALL·E 2. Hãy sử dụng các mô hình `gpt-image` cho các ứng dụng mới.

Trên Microsoft Foundry, **`gpt-image-2`** là mô hình ảnh mới nhất và có khả năng nhất, được đề xuất làm mặc định. Các mô hình `gpt-image-1.5` và `gpt-image-1-mini` cũng có sẵn rộng rãi.

> **Quan trọng:** các mô hình `gpt-image` trả về ảnh được tạo dưới dạng **base64** (`b64_json`), không phải URL. Mã của bạn sẽ giải mã chuỗi base64 thành bytes và lưu lại — không có URL hình ảnh để tải xuống.


## Xây dựng ứng dụng tạo ảnh đầu tiên của bạn

Vậy cần những gì để xây dựng một ứng dụng tạo ảnh? Bạn cần các thư viện sau:

- **python-dotenv**, bạn được khuyên nên sử dụng thư viện này để giữ các thông tin bí mật trong tệp *.env* tách biệt khỏi mã nguồn.
- **openai**, thư viện này là thứ bạn sẽ dùng để tương tác với API OpenAI.
- **pillow**, để làm việc với hình ảnh trong Python.

Nếu chưa làm, hãy làm theo hướng dẫn trên trang [Microsoft Learn](https://learn.microsoft.com/azure/ai-foundry/openai/how-to/create-resource?pivots=web-portal&WT.mc_id=academic-105485-koreyst) để tạo tài nguyên và mô hình Microsoft Foundry. Chọn **gpt-image-2** làm mô hình (mô hình tạo ảnh Azure OpenAI mới nhất; DALL·E là mô hình cũ).

1. Tạo một tệp *.env* với nội dung sau:

    ```text
    AZURE_OPENAI_ENDPOINT=<your endpoint>
    AZURE_OPENAI_API_KEY=<your key>
    AZURE_OPENAI_DEPLOYMENT="gpt-image-2"
    ```

    Tìm thông tin này trong cổng Microsoft Foundry cho tài nguyên của bạn trong phần "Deployments".



1. Thu thập các thư viện trên vào một tệp có tên *requirements.txt* như sau:

    ```text
    python-dotenv
    openai
    pillow
    ```


1. Tiếp theo, tạo môi trường ảo và cài đặt các thư viện:


In [ ]:
# create virtual env
! python3 -m venv venv
# activate environment
! source venv/bin/activate
# install libraries
# pip install -r requirements.txt, if using a requirements.txt file 
! pip install python-dotenv openai pillow

> [!NOTE]
> Đối với Windows, sử dụng các lệnh sau để tạo và kích hoạt môi trường ảo của bạn:

    ```bash
    python3 -m venv venv
    venv\Scripts\activate.bat
    ````

1. Thêm đoạn mã sau vào file có tên *app.py*:

    ```python
    import os
    import base64
    from PIL import Image
    import dotenv
    from openai import AzureOpenAI, BadRequestError

    # import dotenv
    dotenv.load_dotenv()

    # cấu hình client Azure OpenAI (Microsoft Foundry).
    # Các mô hình hình ảnh cần phiên bản API mới — kiểm tra tài liệu Foundry để biết phiên bản cần thiết cho mô hình của bạn.
    client = AzureOpenAI(
      azure_endpoint = os.environ["AZURE_OPENAI_ENDPOINT"],
      api_key=os.environ['AZURE_OPENAI_API_KEY'],
      api_version = "2025-04-01-preview"
      )

    try:
        # Tạo hình ảnh bằng cách sử dụng API tạo hình ảnh.
        generation_response = client.images.generate(
            prompt='Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils',    # Nhập văn bản yêu cầu của bạn ở đây.
            size='1024x1024',
            n=1,
            model=os.environ['AZURE_OPENAI_DEPLOYMENT']   # ví dụ "gpt-image-2"
        )
        # Đặt thư mục để lưu hình ảnh.
        image_dir = os.path.join(os.curdir, 'images')

        # Nếu thư mục không tồn tại, tạo nó.
        if not os.path.isdir(image_dir):
            os.mkdir(image_dir)

        # Khởi tạo đường dẫn hình ảnh (lưu ý định dạng file nên là png).
        image_path = os.path.join(image_dir, 'generated-image.png')

        # Các mô hình gpt-image trả hình ảnh dưới dạng base64 (b64_json), không phải URL.
        image_b64 = generation_response.data[0].b64_json
        generated_image = base64.b64decode(image_b64)
        with open(image_path, "wb") as image_file:
            image_file.write(generated_image)

        # Hiển thị hình ảnh trong trình xem hình ảnh mặc định.
        image = Image.open(image_path)
        image.show()

    # bắt ngoại lệ.
    except BadRequestError as err:
        print(err)

    ```

Hãy cùng giải thích đoạn mã này:

- Đầu tiên, chúng ta nhập các thư viện cần thiết, bao gồm thư viện OpenAI, thư viện dotenv, module base64 và thư viện Pillow.

    ```python
    import os
    import base64
    from PIL import Image
    import dotenv
    from openai import AzureOpenAI, BadRequestError
    ```

- Tiếp theo, chúng ta tải các biến môi trường từ file *.env*.

    ```python
    # nhập dotenv
    dotenv.load_dotenv()
    ```

- Sau đó, chúng ta cấu hình client Azure OpenAI (Microsoft Foundry).

    ```python
    client = AzureOpenAI(
      azure_endpoint = os.environ["AZURE_OPENAI_ENDPOINT"],
      api_key=os.environ['AZURE_OPENAI_API_KEY'],
      api_version = "2025-04-01-preview"
      )
    ```

- Tiếp theo, chúng ta tạo hình ảnh:

    ```python
    generation_response = client.images.generate(
        prompt='Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils',    # Nhập văn bản yêu cầu của bạn tại đây
        size='1024x1024',
        n=1,
        model=os.environ['AZURE_OPENAI_DEPLOYMENT']
    )
    ```

    Các mô hình `gpt-image` trả về hình ảnh dưới dạng chuỗi **base64** trong `data[0].b64_json`. Chúng ta giải mã nó thành bytes và ghi vào file — không có URL để tải xuống.

    ```python
    image_b64 = generation_response.data[0].b64_json
    generated_image = base64.b64decode(image_b64)
    with open(image_path, "wb") as image_file:
        image_file.write(generated_image)
    ```

- Cuối cùng, chúng ta mở hình ảnh và sử dụng trình xem ảnh tiêu chuẩn để hiển thị nó:

    ```python
    image = Image.open(image_path)
    image.show()
    ```

### Thêm chi tiết về việc tạo hình ảnh

Hãy xem các tham số của `images.generate`:

- **prompt**, là đoạn mô tả văn bản dùng để tạo hình ảnh. Ở đây là "Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils".
- **size**, là kích thước của hình ảnh được tạo (ví dụ `1024x1024`, `1536x1024`, `1024x1536`, hoặc `"auto"`).
- **n**, là số lượng hình ảnh được tạo ra. Ở đây chúng ta tạo một ảnh.
- **model**, là tên triển khai mô hình ảnh của bạn (ví dụ `gpt-image-2`).

> Các mô hình hình ảnh không sử dụng tham số `temperature` — tham số này dành cho sinh văn bản. Để có sự đa dạng, gọi API thêm lần nữa; để giảm sự đa dạng, làm prompt cụ thể hơn.

## Các khả năng bổ sung của việc tạo hình ảnh

Bạn đã thấy cách tạo hình ảnh chỉ với vài dòng Python. Các mô hình `gpt-image` cũng có thể **chỉnh sửa** một hình ảnh có sẵn. Bằng cách cung cấp một hình ảnh, thêm một **mask** tùy chọn (đánh dấu khu vực cần thay đổi), và một prompt, bạn có thể thay đổi một phần hình ảnh — ví dụ thêm một chiếc mũ cho con thỏ của chúng ta.

```python
response = client.images.edit(
  model=os.environ['AZURE_OPENAI_DEPLOYMENT'],
  image=open("base_image.png", "rb"),
  mask=open("mask.png", "rb"),
  prompt="An image of a rabbit with a hat on its head.",
  n=1,
  size="1024x1024"
)
# các chỉnh sửa cũng được trả về dưới dạng base64
edited_image = base64.b64decode(response.data[0].b64_json)
```

Hình ảnh gốc chỉ có con thỏ; hình ảnh cuối cùng có thêm chiếc mũ trên con thỏ.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Tuyên bố miễn trừ trách nhiệm**:
Tài liệu này đã được dịch bằng dịch vụ dịch thuật AI [Co-op Translator](https://github.com/Azure/co-op-translator). Mặc dù chúng tôi cố gắng đảm bảo độ chính xác, xin lưu ý rằng bản dịch tự động có thể chứa lỗi hoặc sai sót. Tài liệu gốc bằng ngôn ngữ gốc nên được coi là nguồn tin chính thức. Đối với thông tin quan trọng, nên sử dụng dịch vụ dịch thuật chuyên nghiệp bởi con người. Chúng tôi không chịu trách nhiệm về bất kỳ hiểu lầm hoặc giải thích sai nào phát sinh từ việc sử dụng bản dịch này.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
